# Comparaison d'algos en régression

Nous allons comparer les différentes méthodes de régression que nous avons vues jusqu'à présent, les moindres carrés avec toutes les variables, puis les moindres carrés avec les variables sélectionnées par le critère BIC avec un algo backward et par AIC.
Le nombre de bloc de la validation croisée vaut $k=10$ mais peut être modifié par l'utilisateur. Les données s'appellent *don* et la variable d'intérêt $Y$

In [1]:
import pandas as pd; import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from patsy import dmatrix
import ols_step_sk
###
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.linear_model import RidgeCV, ElasticNetCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
###
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_squared_error

In [180]:
don = pd.read_csv("https://regression-avec-python.github.io/donnees/ozone.txt", header=0, sep=";",index_col=0)
don.rename(columns={"O3":"Y"},inplace=True)

Codage des variables qualitatives nebu et vent

In [3]:
don = pd.read_csv("https://regression-avec-python.github.io/donnees/ozone_transf.txt", header = 0, sep = ";", index_col=0)
print(don.shape)
don.rename(columns={"maxO3":"Y"},inplace=True)

(1366, 22)


In [4]:
nomsvar = list(don.columns.difference(["Y"]))
print(nomsvar)
#design matrix
formule = "~" + "+".join(nomsvar)
print(formule)
dsX = dmatrix(formule,don)
X = np.asarray(dsX)[:,1:]
Y = don["Y"].to_numpy()

['Ne12', 'Ne15', 'Ne18', 'Ne6', 'Ne9', 'T12', 'T15', 'T18', 'T6', 'T9', 'Vx12', 'Vx15', 'Vx18', 'Vx6', 'Vx9', 'Vy12', 'Vy15', 'Vy18', 'Vy6', 'Vy9', 'maxO3v']
~Ne12+Ne15+Ne18+Ne6+Ne9+T12+T15+T18+T6+T9+Vx12+Vx15+Vx18+Vx6+Vx9+Vy12+Vy15+Vy18+Vy6+Vy9+maxO3v


In [5]:
nb=10 ###nb de blocs
tmp = np.arange(don.shape[0])%nb
rng = np.random.default_rng(seed=1234)
bloc = rng.choice(tmp,size=don.shape[0],replace=False) ### l'indice d'appartenance aux blocs

PREV = pd.DataFrame({"bloc":bloc,"Y":don["Y"],"MCO":0.0,"BIC":0.0,"AIC":0.0,
                    "lasso":0.0,"ridge":0.0,"elastic":0.0,"ridgeGS":0.0,'arbre':0.0,'foret':0.0})

In [ ]:
cr = StandardScaler()
kf = KFold(n_splits=10, shuffle=True, random_state=0)
lassocv = LassoCV(cv=kf)
pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
etape_lassocv = pipe_lassocv.named_steps["lassocv"]
lambdaoptlasso=[]
enetcv = ElasticNetCV(cv=kf,max_iter=10000)
pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
ridge = Ridge()
pipe_ridge = Pipeline(steps=[("cr", cr), ("ridge", ridge)])

# PCR : ACP + régression linéaire sur les composantes
acp = PCA()
reg_pcr = LinearRegression()
pipe_pcr = Pipeline(steps=[("cr", cr), ("acp", acp), ("reg", reg_pcr)])
param_grid_pcr = {"acp__n_components": list(range(1, nbaxes))}   # on teste sur 1 à 19 composantes → le meilleur nombre sera choisi par CV

# PLS : maximise la covariance entre X et Y dans l'espace réduit
regpls = PLSRegression()
param_grid_pls = {"n_components": list(range(1, nbaxes))}

In [12]:
for i in np.arange(nb):
    print(i)
    Xapp = X[bloc!=i,:]
    Xtest = X[bloc==i,:]
    Yapp = don[bloc!=i]["Y"]
    Ytest = don[bloc==i]["Y"]
    #### reg
    reg = LinearRegression()
    reg.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"MCO"] = reg.predict(Xtest)
    ### bic
    #inst_reg_bic = ols_step_sk.LinearRegressionSelectionFeatureIC(verbose=1,crit="bic")
    #reg_bic = inst_reg_bic.fit(X=Xapp, y=Yapp)
    #PREV.loc[PREV.bloc==i,"BIC"] = reg_bic.predict(Xtest)
    ### aic
    #inst_reg_aic = ols_step_sk.LinearRegressionSelectionFeatureIC(verbose=1,crit="aic")
    #reg_aic = inst_reg_aic.fit(X=Xapp, y=Yapp)
    #PREV.loc[PREV.bloc==i,"AIC"] = reg_aic.predict(Xtest)
    ###lasso
    pipe_lassocv.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"lasso"] = pipe_lassocv.predict(Xtest)
    ###elastic net pondération 1/2
    pipe_enetcv.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"elastic"] = pipe_enetcv.predict(Xtest)
    ###ridge avec le chemin de reg qui vaut 100 le chemin de lasso
    alphasridge = 100*etape_lassocv.alphas_
    ridgecv = RidgeCV(cv=kf,alphas=alphasridge)
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"ridge"] = pipe_ridgecv.predict(Xtest)
    ## ridge grid searchparams lambda
    path_ridge = alphasridge   
    param_grid_ridge = {"ridge__alpha": path_ridge}
    cv_ridge = GridSearchCV(pipe_ridge, param_grid_ridge, cv=kf, scoring = "neg_mean_squared_error", n_jobs=3).fit(Xapp, Yapp)
    PREV.loc[PREV.bloc==i,"ridgeGS"] = cv_ridge.predict(Xtest)
    ###arbre
    arbre = DecisionTreeRegressor(min_samples_leaf=5).fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"arbre"] = arbre.predict(Xtest)
    ###foret
    foret = RandomForestRegressor().fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"foret"] = foret.predict(Xtest)

0
1
2
3
4
5
6
7
8
9


On a tout préparer, on peut envoyer

In [13]:
prev = PREV.iloc[:,1:]
np.round((prev.sub(PREV.Y, axis=0)**2).mean(),2)

Y             0.00
MCO         187.51
BIC        7745.43
AIC        7745.43
lasso       187.05
ridge       187.45
elastic     187.20
ridgeGS     187.43
arbre       249.35
foret       155.83
dtype: float64

In [97]:
print(lambdaoptlasso)

[2.3165, 1.6083, 1.699, 1.7739, 1.9576, 0.2346, 2.117, 2.2769, 2.3923, 1.5825]
